In [21]:
#Práctica 7 - Simulacion de un sistema criptográfico Mixto

#Imports

import math
from decimal import *
from sympy.ntheory.factor_ import totient

In [22]:
#Datos iniciales (modificar)

alf = "ABCDEFGHIJKLMNÑOPQRSTUVWXYZ ÁÉÍÓÚ"
msj = "CADA VEZ QUE CIFRO CAMBIO LA CLAVE"

#Datos de variante de Vigenere 
K="ENIGMA" #Clave inicial de variante Vigenere

#Datos de RSA por bloques (clave)
#Clave publica
n = 9641865053
e = 70241161

#Susituir los \n por doble espacio
msj = msj.replace('\\r\\n', '  ')
msj = msj.replace('\\n', '  ')
msj = msj.replace('\\\\n', '  ') 


#Otros datos
N = len(alf) #Simbolos del alfabeto
k = math.floor(math.log(n, N)) #Longitud del bloque
getcontext().prec = 50

In [23]:
#Funciones

def obtenerClaveExtendida(claveOriginal):
    clave =[] #valores de la ecuacion de recurrencia ej: 2x + 3y + ... siendo los valores de clave los coeficientes de x, y..
    claveExt =[] #incializado a clave y calculando los siguientes a partir de los len(clave) (tamaño clave) anteriores

    #Añado los valores de la ecuacion de recurrencia para calcular el siguiente
    for c in claveOriginal:
        ci = alf.index(c)
        clave.append(ci)
        claveExt.append(ci)
    
    #Mientras el tamaño de la clave extendida no alcance len(msj) sigo calculando valores
    #Utilizo los len(clave) anteriores valores que serán desde el primero el valor de x
    #hasta el ultimo valor el de la ultima variable
    #Valores mod len(alf)
    while(len(claveExt)<len(msj)):
        nclav = 0
        iteraciones = 0

        for i in reversed(claveExt):
            nclav = nclav + clave[(len(clave)-(iteraciones+1))] * i
            iteraciones = iteraciones +1
            #Si ya he recorrido todas los valores de la clave normal
            if(iteraciones+1 > len(clave)):
                break
        claveExt.append(nclav%N)
    
    return claveExt

##FUNCIONES DE RSA POR BLOQUES

#Funcion que divide el numero entero hasta obtener bloques de longitud K
def obtenerMensaje(c, N, kpo=0):
    msj_final =""

    cociente = 0
    resto = 0

    bucle = True

    while(bucle):

        cociente = math.floor(c/N)
        resto = c%N
        if(cociente < N):
            bucle = False
        else:
            msj_final = msj_final + alf[resto]
            c = cociente

    msj_final = msj_final + alf[resto] + alf[cociente]
    #Si el bloque no llega a la longitud le añado ceros por la izquierda
    if(kpo!=0):
        while(len(msj_final)<kpo):
            msj_final=msj_final+alf[0]

    return msj_final[::-1]

#Funcion que obtiene la codificacion numerica de un caracter en RSA
def obtenerEntero(msg, k):
    num = 0
    for i in range(k):
        num = num + (alf.index( msg[i] ) * pow(N, k-(i+1)))

    return num
    

In [24]:
#Calculo la clave privada (RSA por bloques)
phi = totient(n) #Funcion de sympy para calcular phi
d = pow(e, -1, phi)

In [25]:
#Cifrado

#Obtengo la clave extendida (Vigenere)
Kext = obtenerClaveExtendida(K)

#Cifrar el mensaje con Vigenere con clave extendida

msjNum = [] #mensaje con codificacion numerica
for c in msj:
    msjNum.append(alf.index(c))


#Cifro SUMANDO el mensaje con la clave extendida
msjCifradoNumerico = []
for i in range(len(msj)):
    msjCifradoNumerico.append( (msjNum[i]+Kext[i])%N)

#Mensaje cifrado a texto
msjCifrado =""
for i in msjCifradoNumerico:
    msjCifrado = msjCifrado + alf[i]

#Cifrar la clave (no extendida, K) de Vigenere con RSA por bloques, utilizando la clave PÚBLICA del RECEPTOR

#Bloques de longitud k 
bloques = [K[i:i+k] for i in range(0, len(K), k)]
claveExtendidaCifrada = ""

#Cifro cada bloque de longitud k
for i in range(len(bloques)):
    
    m = obtenerEntero(bloques[i], len(bloques[i])) #Entero a cifrar
    #Cifro m con RSA simple
    c = pow(m, e, n) #Entero cifrado
    

    msj_bloque = obtenerMensaje(c, N, k+1)
    claveExtendidaCifrada = claveExtendidaCifrada + msj_bloque

#Resultados

print("MENSAJE CIFRADO: ")
print("("+claveExtendidaCifrada+" , "+msjCifrado+")")


MENSAJE CIFRADO: 
(BÁÍAEMC , GNLGGVEFCLGKÓVYÚÉQHOOREVÚNÚKÍQÉAFÑ)


In [26]:
# #Descifrado
# 
# #Descifrar la clave de Vigenere con RSA por bloques utilizando la clave PRIVADA del RECEPTOR
# 
# #Bloques de longitud k+1
# bloques = [K[i:i+(k+1)] for i in range(0, len(K), (k+1))]
# clave_vig_descifrado = ""
# 
# #Descifro cada bloque de longitud k+1
# for i in range(len(bloques)):
# 
#     c = obtenerEntero(bloques[i], len(bloques[i])) #Entero a descifrar
#     #Descifro con RSA simple la c
#     m = pow(c, d, n) #Entero descifrado
#     msj_bloque = obtenerMensaje(m, N, k)
#     clave_vig_descifrado = clave_vig_descifrado + msj_bloque
# 
# print("Clave Vigenere: "+clave_vig_descifrado)
# 
# #Descifrar el mensaje con Vigenere con clave extendida
# 
# #Obtengo la clave extendida para el descifrado
# Kext = obtenerClaveExtendida(clave_vig_descifrado)
# 
# msjCifNum = [] #mensaje para descifrar con codificacion numerica
# for c in msj:
#     msjCifNum.append(alf.index(c))
# 
# #Descifro RESTANDO el mensaje con la clave extendida
# msjDescifradoNumerico = []
# for i in range(len(msjCifNum)):
#     msjDescifradoNumerico.append( (msjCifNum[i]-Kext[i])%N)
# 
# #Mensaje cifrado a texto
# msjDescifrado =""
# for n in msjDescifradoNumerico:
#     msjDescifrado = msjDescifrado + alf[n]
# 
# print("")
# print("MENSAJE DESCIFRADO: ")
# print(msjDescifrado)